# 效率与局限性

```{admonition} 学习目标
阅读本章后，您将能够：
- 设计实验，测量差分隐私算法的时间和空间开销
- 权衡空间效率与时间效率
- 考虑可用于优化的技术
- 描述差分隐私算法的效率瓶颈
- 描述差分隐私的局限性
```

In [ ]:
%matplotlib inline
from mplfonts import use_font
use_font('SimHei')
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import pandas as pd
import numpy as np
from collections import defaultdict

# 一些有用的工具函数

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon, size=len(v))

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

def z_clip(xs, b):
    return [min(x, b) for x in xs]

def g_clip(v):
    n = np.linalg.norm(v, ord=2)
    if n > 1:
        return v / n
    else:
        return v
    
X = np.load('adult_processed_x.npy')
y = np.load('adult_processed_y.npy')

training_size = int(X.shape[0] * 0.8)

X_train = X[:training_size]
X_test = X[training_size:]

y_train = y[:training_size]
y_test = y[training_size:]

y_test.shape

def predict(xi, theta, bias=0):
    label = np.sign(xi @ theta + bias)
    return label

# 损失函数用于衡量我们的模型有多好。训练目标是最小化损失值。
# 这是对率损失函数。
def loss(theta, xi, yi):
    exponent = - yi * (xi.dot(theta))
    return np.log(1 + np.exp(exponent))

# 这是对率损失函数的梯度
# 梯度是一个表示各个方向损失变化率的向量
def gradient(theta, xi, yi):
    exponent = yi * (xi.dot(theta))
    return - (yi*xi) / (1+np.exp(exponent))

def accuracy(theta):
    return np.sum(predict(X_test, theta) == y_test)/X_test.shape[0]

def avg_grad(theta, X, y):
    grads = [gradient(theta, xi, yi) for xi, yi in zip(X, y)]
    return np.mean(grads, axis=0)

def gradient_descent(iterations):
    # 我们用"猜测"的一个模型参数（权重全为0的模型）作为起始点
    theta = np.zeros(X_train.shape[1])

    # 应用训练集执行`iterations`步梯度下降
    for i in range(iterations):
        theta = theta - avg_grad(theta, X_train, y_train)

    return theta

def L2_clip(v, b):
    norm = np.linalg.norm(v, ord=2)
    
    if norm > b:
        return b * (v / norm)
    else:
        return v
    
def gradient_sum(theta, X, y, b):
    gradients = [gradient(theta, x_i, y_i) for x_i, y_i in zip(X,y)]
        
    # 求和问询
    # L2敏感度为b（由梯度的敏感度决定）
    return np.sum(gradients, axis=0)

def noisy_gradient_descent(iterations, epsilon, delta):
    theta = np.zeros(X_train.shape[1])
    sensitivity = 5.0
    
    noisy_count = laplace_mech(X_train.shape[0], 1, epsilon)
    clipped_X = [L2_clip(x_i, sensitivity) for x_i in X_train]

    for i in range(iterations):
        grad_sum        = gradient_sum(theta, clipped_X, y_train, sensitivity)
        noisy_grad_sum  = gaussian_mech_vec(grad_sum, sensitivity, epsilon, delta)
        noisy_avg_grad  = noisy_grad_sum / noisy_count
        theta           = theta - noisy_avg_grad

    return theta

## 差分隐私的时间效率

如果差分隐私是通过直接遍历敏感实体的循环来实现的，那么生成随机数所带来的额外负担会产生显著的时间开销。

我们可以对比两个版本计数问询的运行时间性能：一个满足差分隐私，另一个不满足差分隐私。

In [ ]:
import itertools
import operator
import time

def time_count(k):   
    l = [1] * k
    start = time.perf_counter()
    _ = list(itertools.accumulate(l, func=operator.add))
    stop = time.perf_counter()
    return stop - start 

def time_priv_count(k):  
    l = [1] * k
    start = time.perf_counter()
    _ = list(itertools.accumulate(l, func=lambda x, y: x + laplace_mech(y,1,0.1), initial=0))
    stop = time.perf_counter()
    return stop - start 

In [ ]:
x_axis = [k for k in range(100_000,1_000_000,100_000)]
plt.xlabel('规模（N）')
plt.ylabel('时间（秒）')
plt.plot(x_axis, [time_count(k) for k in range(100_000,1_000_000,100_000)], label='普通计数')
plt.plot(x_axis, [time_priv_count(k) for k in range(100_000,1_000_000,100_000)], label='差分隐私计数')
plt.legend();

图中$y$轴表示计数所花费的时间，$x$轴表示列表大小。可以看出，对于这一特定操作，差分隐私的时间复杂度基本上与输入规模呈线性关系。

好消息是，上述差分隐私计数的实现方法相当朴素。我们可以使用向量化等优化技术得到更好的实现！

例如，在[前面](ch12.ipynb)章节中为机器学习操作实现差分隐私时，我们使用了针对向量运算进行过深度优化的NumPy函数。在这一场景下，利用这种策略，使用差分隐私所带来的时间开销几乎可以忽略不计。

In [ ]:
delta = 1e-5

def time_gd(k):   
    start = time.perf_counter()
    gradient_descent(k)
    stop = time.perf_counter()
    return stop - start 

def time_priv_gd(k):   
    start = time.perf_counter()
    noisy_gradient_descent(k, 0.1, delta)
    stop = time.perf_counter()
    return stop - start 

In [ ]:
x_axis = [k for k in range(10,100,25)]
plt.xlabel('规模（N）')
plt.ylabel('时间（秒）')
plt.plot(x_axis, [time_gd(k) for k in range(10,100,25)], label='普通梯度下降')
plt.plot(x_axis, [time_priv_gd(k) for k in range(10,100,25)], label='差分隐私梯度下降')
plt.legend();

在这一实验的大多数模拟结果中，两条曲线基本重合或非常接近，这表明使用差分隐私梯度下降所带来的时间开销很低（为常数）。

## 差分隐私下模型训练所需迭代次数的增加

为了满足差分隐私而在模型训练中引入噪声时，往往需要更多的迭代次数（如更多的训练步数或训练轮数），模型才能收敛到可接受的性能水平。

训练时间的增加主要源于噪声对学习过程的干扰。DP-SGD（差分隐私随机梯度下降，Differentially Private Stochastic Gradient Descent）等差分隐私机制在优化过程中为梯度添加噪声。这些梯度是根据小批量数据计算得到的，添加噪声的目的是掩盖任意单个数据点的影响。虽然这种方法可以有效地保护隐私，但其副作用是让梯度的噪声更大、包含的信息更少。优化器接收到的、关于损失函数曲面中真实移动方向的信号更弱，这可能会减慢学习速度。

为了应对噪声带来的不稳定性，从业者通常会使用更小的学习率。更小的步长有助于避免模型出现异常行为，但也会导致收敛速度变慢。梯度的可靠性降低，更新步骤也更加谨慎，两者叠加意味着模型通常需要更长的时间才能得到有用的解。在很多情况下，这意味着需要训练更多轮数、评估更多检查点，并可能需要使用容忍度更宽松的早停策略。

在实际应用中，这意味着在差分隐私下训练模型不仅需要更多的迭代次数，还需要更仔细地调整学习率、批量大小、裁剪范数和噪声乘数等超参数。例如，本书前面对比了满足和不满足差分隐私的逻辑回归模型训练过程，并观察到满足差分隐私的模型收敛得更慢，能达到的最高准确率也更低。这与文献中的观察结果一致：差分隐私让学习变得更加困难，但只要采用正确的策略并投入足够的计算资源，仍然可以训练出有用的模型。

## 差分隐私的空间开销

我们也可以分析差分隐私机制的空间使用情况。Python3（3.4）引入了一个[调试工具](https://docs.python.org/3/library/tracemalloc.html)，可以追踪程序执行期间分配的内存块。我们可以比较差分隐私计算和非差分隐私计算所使用内存块的峰值大小，以此分析空间开销。

利用这一策略，我们可以按照下述方法分析差分隐私计数操作的空间开销：

In [ ]:
import itertools
import operator
import tracemalloc

def space_count(k):   
    l = [1] * k
    itertools.accumulate(l, func=operator.add)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.reset_peak()
    return peak/1000 

def space_priv_count(k):  
    l = [1] * k
    itertools.accumulate(l, func=lambda x, y: x + laplace_mech(y,1,0.1), initial=0)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.reset_peak()
    return peak/1000

In [ ]:
tracemalloc.start()
x_axis = [k for k in range(100_000,1_000_000,100_000)]
plt.xlabel('规模（N）')
plt.ylabel('内存块')
plt.plot(x_axis, [space_count(k) for k in range(100_000,1_000_000,100_000)], label='普通计数')
plt.plot(x_axis, [space_priv_count(k) for k in range(100_000,1_000_000,100_000)], label='差分隐私计数')
tracemalloc.stop()
plt.legend();

图中$y$轴以千为单位表示内存块数量，$x$轴表示列表大小。

在这种情况下，差分隐私的空间复杂度基本上是常数。

我们还观察到，差分隐私计数的初始空间开销出现了一个峰值。这可以归因于为设置随机数生成器相关资源（如熵池和PRNG，即伪随机数生成器）而分配的内存。

我们可以重复这一实验，对比差分隐私和非差分隐私的机器学习：

In [ ]:
def space_gd(k):   
    gradient_descent(k)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.reset_peak()
    return peak/1_000_000 

def space_priv_gd(k):   
    noisy_gradient_descent(k, 0.1, delta)
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.reset_peak()
    return peak/1_000_000

In [ ]:
tracemalloc.start()
x_axis = [k for k in range(5,10,2)]
plt.xlabel('规模（N）')
plt.ylabel('内存块')
plt.plot(x_axis, [space_gd(k) for k in range(5,10,2)], label='普通梯度下降')
plt.plot(x_axis, [space_priv_gd(k) for k in range(5,10,2)], label='差分隐私梯度下降')
tracemalloc.stop()
plt.legend();

在这种情况下，由于我们执行的是梯度下降，内存分配量通常要大得多（这符合预期），因此图中以百万为单位表示已分配的内存块数量。

我们同样观察到，差分隐私梯度下降的空间开销基本上是常数。不过，这里的空间开销要大得多，因为我们为了提高时间效率，额外使用了一些空间来实现向量化和并发。


```{note}

### 差分隐私中的幽灵裁剪

**幽灵裁剪**（Ghost Clipping）是一种用于提高差分隐私模型训练时间效率和空间效率的技术，特别适用于差分隐私深度学习中的**差分隐私大语言模型**（Large Language Models，LLMs）训练。

幽灵裁剪不在训练过程中直接实例化和修改每个样本的梯度，而是直接基于每个样本的激活梯度和每个样本的激活值进行计算，从而避免了实例化梯度的需要。

Meta的[Opacus](https://opacus.ai/)和谷歌的[TensorFlow Privacy](https://www.tensorflow.org/responsible_ai/privacy/guide)等工具都使用了这一技术。

您可以阅读[相关学术论文](https://arxiv.org/pdf/2110.05679)和[PyTorch博客文章](https://pytorch.org/blog/clipping-in-opacus/)，进一步了解幽灵裁剪。


```



## 随机数生成的局限性

总而言之，虽然可能存在一些瓶颈，但我们可以利用一些技术，在程序执行的整个生命周期中让差分隐私的空间开销和时间开销总体上保持在较低（常数）水平。

为什么会存在瓶颈？为什么这些瓶颈是差分隐私所特有的？为了保证差分隐私，我们需要高质量的熵。熵是一种有限的资源，因为它通常依赖于某种外部的非确定性环境输入。非确定性熵源的例子包括磁盘/网络IO、键盘按键和鼠标移动。

这些熵通常存储在某个特殊的[缓冲区或文件](https://en.wikipedia.org/wiki/%2Fdev%2Frandom)中，以便后续获取和使用。

In [ ]:
with open("/dev/random", 'rb') as file:
    print(file.read(8))

或者，我们也可以使用：

In [ ]:
import os
os.urandom(32)

或者：

In [ ]:
import secrets 
secrets.token_bytes(32)

一般来说，不同的编程语言和操作系统提供了多种生成随机数和随机种子的方法。这些方法适用于不同的应用场景，例如建模、仿真，或生成适合大规模管理敏感用户数据的密码学安全随机字节。

## 实数与随机噪声

差分隐私依赖于随机化机制，这些机制添加的噪声来自连续分布——最常见的是**拉普拉斯**分布或**高斯**分布。这些分布定义在实数上，但计算机无法精确表示实数。我们使用的是**浮点数近似**，这可能会引入一些微妙但严重的问题。

在实际应用中，当我们生成随机噪声时（例如使用NumPy中的`np.random.laplace()`），我们依赖的是有限精度的算术运算。虽然这在大多数应用场景下都足够好用，但它可能会导致分布**尾部出现确定性行为**，或产生可被攻击者利用的微小偏差。

Ilya Mironov在一篇里程碑式的论文中指出，使用浮点数算术运算朴素地实现拉普拉斯机制，可能会通过所生成噪声值中的细微瑕疵**泄露隐私信息**——这主要是由于可表示浮点数的间隔不均匀所导致的。这些瑕疵可能会使输出分布中出现**不连续性或特定模式**，而聪明的攻击者可以检测并利用这些不连续性或模式{cite}`mironov2012`。

为了降低这些风险，学者们提出了下述几种策略：

- **精心设计的采样算法**，如Mironov提出的*对齐机制*（Snapping Mechanism）。该机制将输出对齐到均匀的浮点数网格上，同时仍然满足差分隐私。
- 使用**离散噪声分布**（如几何分布或离散拉普拉斯分布）作为近似。在有限精度的场景下，离散噪声分布更易于分析。
- 使用**密码学安全的随机源**，以避免噪声中出现确定性瑕疵。

### 实践建议

对于大多数实际实现来说，了解这些问题非常重要，特别是在需要强形式化保证的情况下。[谷歌的差分隐私库](https://github.com/google/differential-privacy)和[IBM的diffprivlib](https://github.com/IBM/differential-privacy-library)等工具库都采取了防御措施来应对这些弱点，它们的策略是实现稳健部署的良好范例。